In [1]:
import os
from pathlib import Path

HOME_DIR = os.path.dirname(os.getcwd())
DATA_DIR = Path("/data/dacon-236105")
MODEL_DIR = Path(HOME_DIR) / "model"
OUTPUT_DIR = Path(HOME_DIR) / "output"
TRAIN_FEAT_DIR = DATA_DIR / "train_whisper"   # {TRAIN_FEAT_DIR}/layer{l:02d}/{id}.npy
TEST_FEAT_DIR = DATA_DIR / "test_whisper"     # {TEST_FEAT_DIR}/layer{l:02d}/{id}.npy

MODEL_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

# EDA 3. Whisper large-v3 Encoder Layer-wise Features → BiLSTM

- `openai/whisper-large-v3`의 **encoder 가중치만** safetensors에서 읽어 GPU에 올림 (decoder는 메모리에 아예 로드하지 않음)
- 32개 encoder layer 출력(hidden_states[1..32])을 실제 오디오 길이만큼만 잘라 layer별 `.npy`(float16)로 저장
  - Whisper 입력은 30 s로 zero-padding되므로, 유효 encoder frame 수 = ceil(ceil(samples / 160) / 2) (20 ms/frame)
  - hidden_states[32]는 encoder 마지막 `layer_norm`이 적용된 출력 (= `last_hidden_state`)
- layer마다 저장된 feature를 읽어 eda02와 동일한 BiLSTM / 5-fold / AdamW + ReduceLROnPlateau로 학습
- 출력: `output/whisper_layer{l:02d}.csv`, 모델: `model/whisper_layer{l:02d}_fold{k}.pt`

## Import

In [2]:
import gc
import random
import time
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.io import wavfile
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence, pad_sequence
from torch.utils.data import DataLoader, Dataset

from huggingface_hub import hf_hub_download
from safetensors import safe_open
from transformers import WhisperConfig, WhisperFeatureExtractor
from transformers.models.whisper.modeling_whisper import WhisperEncoder

warnings.filterwarnings(action="ignore")

/home/hommy/Repositories/dacon-236105/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Hyperparameter Setting

In [3]:
CFG = {
    # Feature
    "SR": 16000,
    "WHISPER_ID": "openai/whisper-large-v3",
    "LAYERS": [1, 4, 8, 12, 16, 20, 24, 28, 32],  # hidden_states index: 0 = conv+pos embedding, 1..32 = encoder layers
    "EXTRACT_BATCH_SIZE": 8,

    # Model
    "N_CLASSES": 6,
    "HIDDEN": 256,
    "NUM_LAYERS": 2,
    "DROPOUT": 0.3,

    # Training
    "N_FOLDS": 5,
    "BATCH_SIZE": 64,
    "EPOCHS": 500,
    "LR": 1e-4,
    "WEIGHT_DECAY": 1e-2,
    "LR_FACTOR": 0.5,    # ReduceLROnPlateau
    "LR_PATIENCE": 8,    # ReduceLROnPlateau
    "EARLY_STOP": 20,    # best val_acc 대비 20 epoch 연속 개선 없으면 중단 (저장은 best epoch만)
    "GRAD_CLIP": 5.0,
    "SEED": 42,
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

## Fixed Random-Seed

In [4]:
def seed_everything(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(CFG["SEED"]) # Seed 고정

## Data Pre-Processing

In [5]:
train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

train_df["path"] = train_df["path"].apply(lambda p: str(DATA_DIR / p))
test_df["path"] = test_df["path"].apply(lambda p: str(DATA_DIR / p))

labels = train_df["label"].values
len(train_df), len(test_df)

(5001, 1881)

### Whisper Encoder Feature Extraction

EDA 1 결과: 전부 16 kHz / int16 / mono, 길이 1.33–5.0 s → 유효 encoder frame 약 67–251개.
이미 모든 layer 파일이 있는 clip은 건너뛴다 (재실행 시 추출 생략).

In [6]:
def load_wav(path):
    sr, y = wavfile.read(path)
    assert sr == CFG["SR"] and y.dtype == np.int16 and y.ndim == 1, (path, sr, y.dtype, y.shape)
    return y.astype(np.float32) / 32768.0

def load_whisper_encoder(model_id):
    """Encoder-only: decoder 가중치는 읽지도, GPU에 올리지도 않는다."""
    config = WhisperConfig.from_pretrained(model_id)
    ckpt_path = hf_hub_download(model_id, "model.safetensors")
    prefix = "model.encoder."
    with safe_open(ckpt_path, framework="pt") as f:
        state = {k[len(prefix):]: f.get_tensor(k) for k in f.keys() if k.startswith(prefix)}
    with torch.device("meta"):
        encoder = WhisperEncoder(config)
    encoder.load_state_dict(state, strict=True, assign=True)
    return encoder.to(device, torch.float16).eval()

def feat_path(feat_dir, layer, file_id):
    return feat_dir / f"layer{layer:02d}" / f"{file_id}.npy"

@torch.inference_mode()
def extract_whisper_features(df, feat_dir, encoder, feature_extractor):
    for layer in CFG["LAYERS"]:
        (feat_dir / f"layer{layer:02d}").mkdir(parents=True, exist_ok=True)
    # 마지막 layer 파일을 가장 나중에 저장하므로, 그것이 있으면 해당 clip은 완료된 것으로 본다
    todo = df[[not feat_path(feat_dir, CFG["LAYERS"][-1], i).exists() for i in df["id"]]]
    print(f"{feat_dir.name}: {len(df) - len(todo)} done, {len(todo)} to extract")

    bs = CFG["EXTRACT_BATCH_SIZE"]
    for start in tqdm(range(0, len(todo), bs)):
        batch = todo.iloc[start:start + bs]
        wavs = [load_wav(p) for p in batch["path"]]
        inputs = feature_extractor(
            wavs, sampling_rate=CFG["SR"], return_tensors="pt", return_attention_mask=True,
        )
        hidden_states = encoder(
            inputs.input_features.to(device, torch.float16), output_hidden_states=True,
        ).hidden_states  # tuple of 33 x [B, 1500, 1280]
        n_mel_frames = inputs.attention_mask.sum(dim=1)  # = ceil(samples / 160)
        n_enc_frames = ((n_mel_frames + 1) // 2).tolist()  # conv2 stride 2
        for b, file_id in enumerate(batch["id"]):
            n = n_enc_frames[b]
            for layer in CFG["LAYERS"]:
                feat = hidden_states[layer][b, :n].cpu().numpy()
                assert np.isfinite(feat).all(), (file_id, layer)
                np.save(feat_path(feat_dir, layer, file_id), feat)

In [7]:
encoder = load_whisper_encoder(CFG["WHISPER_ID"])
feature_extractor = WhisperFeatureExtractor.from_pretrained(CFG["WHISPER_ID"])
print(f"encoder params: {sum(p.numel() for p in encoder.parameters()) / 1e6:.1f}M | "
      f"GPU allocated: {torch.cuda.memory_allocated() / 2**20:.0f} MiB")

extract_whisper_features(train_df, TRAIN_FEAT_DIR, encoder, feature_extractor)
extract_whisper_features(test_df, TEST_FEAT_DIR, encoder, feature_extractor)

del encoder
gc.collect()
torch.cuda.empty_cache()

encoder params: 637.0M | GPU allocated: 1216 MiB
train_whisper: 5001 done, 0 to extract


0it [00:00, ?it/s]

0it [00:00, ?it/s]

test_whisper: 1881 done, 0 to extract


0it [00:00, ?it/s]

0it [00:00, ?it/s]

In [8]:
def load_layer_feats(df, feat_dir, layer):
    """[T_i, 1280] float16 array list, df 순서 유지."""
    return [np.load(feat_path(feat_dir, layer, i)) for i in tqdm(df["id"], desc=f"layer{layer:02d}", leave=False)]

### Normalization

In [9]:
def fit_normalizer(feats):
    n = sum(len(f) for f in feats)
    s = sum(f.sum(axis=0, dtype=np.float64) for f in feats)
    ss = sum((f.astype(np.float64) ** 2).sum(axis=0) for f in feats)
    mean = s / n
    std = np.sqrt(np.maximum(ss / n - mean ** 2, 1e-12))
    return mean.astype(np.float32), std.astype(np.float32)

## Dataset / DataLoader

In [10]:
class AudioFeatDataset(Dataset):
    def __init__(self, feats, labels=None, mean=None, std=None):
        self.feats = feats
        self.labels = labels
        self.mean = mean
        self.std = std

    def __len__(self):
        return len(self.feats)

    def __getitem__(self, idx):
        x = (self.feats[idx].astype(np.float32) - self.mean) / self.std  # float16 저장 → float32 학습
        y = -1 if self.labels is None else int(self.labels[idx])
        return torch.from_numpy(x), y

def collate_fn(batch):
    xs, ys = zip(*batch)
    lengths = torch.tensor([len(x) for x in xs], dtype=torch.long)
    x_pad = pad_sequence(xs, batch_first=True)  # [B, T_max, D]
    return x_pad, lengths, torch.tensor(ys, dtype=torch.long)

## Model

In [11]:
class AttentionPooling(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.score = nn.Linear(dim, 1)

    def forward(self, x, lengths):
        # x: [B, T, D]
        B, T, D = x.shape
        scores = self.score(x).squeeze(-1)  # [B, T]
        mask = (
            torch.arange(T, device=x.device)[None, :]
            < lengths.to(x.device)[:, None]
        )
        scores = scores.masked_fill(~mask, float("-inf"))
        weights = torch.softmax(scores, dim=1)  # [B, T]
        pooled = torch.sum(
            x * weights.unsqueeze(-1), dim=1
        )  # [B, D]
        return pooled

class BiLSTMClassifier(nn.Module):
    def __init__(self, in_dim, hidden, num_layers, n_classes, dropout):
        super().__init__()
        self.lstm = nn.LSTM(
            in_dim, hidden, num_layers=num_layers, batch_first=True, bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.pool = AttentionPooling(2 * hidden)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(2 * hidden, n_classes))

    def forward(self, x, lengths):
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out, _ = self.lstm(packed)
        out, out_lens = pad_packed_sequence(out, batch_first=True)  # [B, T_max, 2H]
        pooled = self.pool(out, out_lens)  # masked attention pooling
        return self.head(pooled)

## Train / Validation

In [12]:
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    total_loss, probs, ys = 0.0, [], []
    with torch.set_grad_enabled(is_train):
        for x, lengths, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x, lengths)
            loss = criterion(logits, y)
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), CFG["GRAD_CLIP"])
                optimizer.step()
            total_loss += loss.item() * len(y)
            probs.append(logits.softmax(dim=-1).detach().cpu())
            ys.append(y.cpu())
    probs, ys = torch.cat(probs).numpy(), torch.cat(ys).numpy()
    return total_loss / len(ys), probs, ys

@torch.no_grad()
def predict(model, loader):
    model.eval()
    return np.concatenate([
        model(x.to(device), lengths).softmax(dim=-1).cpu().numpy() for x, lengths, _ in loader
    ])

def make_loader(feats, labels, mean, std, shuffle, seed=None):
    return DataLoader(
        AudioFeatDataset(feats, labels, mean, std),
        batch_size=CFG["BATCH_SIZE"], shuffle=shuffle, collate_fn=collate_fn,
        generator=torch.Generator().manual_seed(seed) if shuffle else None,
    )

In [13]:
def run_cv(name, train_feats, test_feats, labels):
    skf = StratifiedKFold(n_splits=CFG["N_FOLDS"], shuffle=True, random_state=CFG["SEED"])
    oof = np.zeros((len(train_feats), CFG["N_CLASSES"]), dtype=np.float32)
    test_prob = np.zeros((len(test_feats), CFG["N_CLASSES"]), dtype=np.float32)
    fold_logs = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(labels)), labels)):
        seed_everything(CFG["SEED"] + fold)
        tr_feats = [train_feats[i] for i in tr_idx]
        va_feats = [train_feats[i] for i in va_idx]
        mean, std = fit_normalizer(tr_feats)

        tr_loader = make_loader(tr_feats, labels[tr_idx], mean, std, shuffle=True, seed=CFG["SEED"] + fold)
        va_loader = make_loader(va_feats, labels[va_idx], mean, std, shuffle=False)
        te_loader = make_loader(test_feats, None, mean, std, shuffle=False)

        model = BiLSTMClassifier(
            train_feats[0].shape[1], CFG["HIDDEN"], CFG["NUM_LAYERS"], CFG["N_CLASSES"], CFG["DROPOUT"],
        ).to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["LR"], weight_decay=CFG["WEIGHT_DECAY"])
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="max", factor=CFG["LR_FACTOR"], patience=CFG["LR_PATIENCE"],
        )

        ckpt_path = MODEL_DIR / f"{name}_fold{fold}.pt"
        best_acc, best_epoch, wait = -1.0, 0, 0
        t0 = time.time()
        for epoch in range(1, CFG["EPOCHS"] + 1):
            tr_loss, _, _ = run_epoch(model, tr_loader, criterion, optimizer)
            va_loss, va_prob, va_y = run_epoch(model, va_loader, criterion)
            va_acc = accuracy_score(va_y, va_prob.argmax(axis=1))
            scheduler.step(va_acc)

            improved = va_acc > best_acc
            if improved:
                best_acc, best_epoch, wait = va_acc, epoch, 0
                torch.save({
                    "model": model.state_dict(),
                    "mean": torch.from_numpy(mean), "std": torch.from_numpy(std),
                    "in_dim": train_feats[0].shape[1], "epoch": epoch, "cfg": CFG,
                }, ckpt_path)
            else:
                wait += 1
            lr = optimizer.param_groups[0]["lr"]
            mark = " *" if improved else ''
            print(f"[{name} | fold {fold}] ep {epoch:02d} | tr_loss {tr_loss:.4f} | va_loss {va_loss:.4f} "
                  f"| va_acc {va_acc:.4f} | lr {lr:.1e}{mark}")
            if wait >= CFG["EARLY_STOP"]:
                break
        train_time = time.time() - t0

        ckpt = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(ckpt["model"])
        oof[va_idx] = predict(model, va_loader)
        test_prob += predict(model, te_loader) / CFG["N_FOLDS"]

        va_pred = oof[va_idx].argmax(axis=1)
        fold_logs.append({
            "fold": fold, "best_epoch": best_epoch,
            "val_acc": accuracy_score(labels[va_idx], va_pred),
            "val_macro_f1": f1_score(labels[va_idx], va_pred, average="macro"),
            "train_time_sec": round(train_time, 1),
        })
        fold_acc = fold_logs[-1]["val_acc"]
        print(f"[{name} | fold {fold}] best ep {best_epoch} | val_acc {fold_acc:.4f}\n")

    fold_logs = pd.DataFrame(fold_logs)
    oof_pred = oof.argmax(axis=1)
    oof_acc = accuracy_score(labels, oof_pred)
    oof_f1 = f1_score(labels, oof_pred, average="macro")
    print(f"[{name}] OOF acc {oof_acc:.4f} | OOF macro-F1 {oof_f1:.4f}")
    return oof, test_prob, fold_logs

## Submission Helper

In [14]:
def save_submission(test_prob, fname):
    submission = pd.read_csv(DATA_DIR / "sample_submission.csv")
    pred = pd.Series(test_prob.argmax(axis=1), index=test_df["id"].values)
    submission["label"] = submission["id"].map(pred)
    assert submission["label"].notna().all()
    submission["label"] = submission["label"].astype(int)
    submission.to_csv(OUTPUT_DIR / fname, index=False)
    return submission

## Layer-wise Training (Whisper large-v3 encoder layer 1–32 → BiLSTM)

layer마다 해당 layer의 `.npy`를 모두 읽어 5-fold 학습 → fold 앙상블로 test 추론 → `output/whisper_layer{l:02d}.csv` 저장.

In [15]:
layer_results = []

for layer in CFG["LAYERS"]:
    name = f"whisper_layer{layer:02d}"
    train_feats = load_layer_feats(train_df, TRAIN_FEAT_DIR, layer)
    test_feats = load_layer_feats(test_df, TEST_FEAT_DIR, layer)

    oof, test_prob, logs = run_cv(name, train_feats, test_feats, labels)
    save_submission(test_prob, f"{name}.csv")
    display(logs)

    oof_pred = oof.argmax(axis=1)
    layer_results.append({
        "layer": layer,
        "oof_acc": accuracy_score(labels, oof_pred),
        "oof_macro_f1": f1_score(labels, oof_pred, average="macro"),
        "fold_acc_mean": logs["val_acc"].mean(),
        "fold_acc_std": logs["val_acc"].std(),
        "best_epoch_mean": logs["best_epoch"].mean(),
        "train_time_sec": logs["train_time_sec"].sum(),
    })

    del train_feats, test_feats, oof, test_prob
    gc.collect()

layer01:   0%|          | 0/5001 [00:00<?, ?it/s]

layer01:  16%|█▌        | 791/5001 [00:00<00:00, 7906.37it/s]

layer01:  32%|███▏      | 1593/5001 [00:00<00:00, 7972.69it/s]

layer01:  48%|████▊     | 2391/5001 [00:00<00:00, 7752.82it/s]

layer01:  65%|██████▍   | 3234/5001 [00:00<00:00, 8012.43it/s]

layer01:  82%|████████▏ | 4093/5001 [00:00<00:00, 8217.75it/s]

... (1,713 outputs left out)

[whisper_layer32 | fold 4] ep 37 | tr_loss 0.0015 | va_loss 1.3889 | va_acc 0.7340 | lr 2.5e-05


[whisper_layer32 | fold 4] ep 38 | tr_loss 0.0014 | va_loss 1.3946 | va_acc 0.7320 | lr 2.5e-05


[whisper_layer32 | fold 4] best ep 18 | val_acc 0.7500

[whisper_layer32] OOF acc 0.7636 | OOF macro-F1 0.7647


,fold,best_epoch,val_acc,val_macro_f1,train_time_sec
0,0,19,0.768232,0.769158,147.5
1,1,14,0.749000,0.751244,129.3
2,2,6,0.778000,0.779635,98.5
3,3,20,0.773000,0.773022,151.4
4,4,18,0.750000,0.750367,143.9


## Comparison

In [16]:
layer_results_df = pd.DataFrame(layer_results)
display(layer_results_df.sort_values("oof_acc", ascending=False))

,layer,oof_acc,oof_macro_f1,fold_acc_mean,fold_acc_std,best_epoch_mean,train_time_sec
5,20,0.796241,0.797232,0.796239,0.014896,13.0,688.2
6,24,0.790842,0.792000,0.790842,0.016938,11.2,650.4
7,28,0.790242,0.791716,0.790245,0.019313,23.0,897.3
4,16,0.777245,0.777508,0.777245,0.008190,19.6,827.2
8,32,0.763647,0.764723,0.763646,0.013372,15.4,670.6
3,12,0.755449,0.755841,0.755447,0.007604,14.6,724.8
2,8,0.722256,0.723144,0.722254,0.010904,18.2,797.0
1,4,0.693261,0.693060,0.693262,0.007844,16.0,746.9
0,1,0.656669,0.655725,0.656665,0.015187,12.8,680.8
